# Night Shift V2: model B

Use this notebook on the **second laptop**. We start from model B at **250,000 V1 steps** and train for **400,000 additional V2 steps**. Hidden layers: `64, 64`.

**Run All does not start training or evaluation.** It loads weights, shows saved history and creates explicit buttons. Use a new kernel to avoid importing V1 code.

The model uses a neural network, epsilon-greedy actions, replay memory, a target network and the Double DQN target. The implementation is in `train_v2.py`; all experiments use the same code.

## 1. Imports and device

Keep the existing CUDA PyTorch installation. If other packages are missing, install `requirements-training.txt` in this Python environment. All paths are relative to the copied V2 folder.

In [ ]:
import os
import sys
from pathlib import Path
os.environ["PYGAME_HIDE_SUPPORT_PROMPT"]="1"

# All files travel together when this folder is copied to another laptop.
candidates=[Path.cwd(),Path.cwd()/"V2",Path.cwd()/"versions"/"V2"]
root=next((p.resolve() for p in candidates if (p/"train_v2.py").exists()),None)
if root is None:
    raise RuntimeError("Open this notebook from the V2 folder.")
for name in ("gym_env","simulation","train_v2"):
    module=sys.modules.get(name)
    if module is not None and Path(module.__file__).resolve().parent!=root:
        raise RuntimeError("Restart the kernel: another game version is already imported.")
if str(root) not in sys.path:
    sys.path.insert(0,str(root))

import json
import torch
import gym_env
from IPython.display import display,Video
from train_v2 import load_policy,show_status,run_training
from eval_v2 import evaluate_initial,evaluate_final,show_evaluations,plot_training,compare_final
assert gym_env.ENV_ID=="NightShift-v2"
print("Python:",sys.executable)
print("PyTorch:",torch.__version__)
print("Device:","cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print("GPU:",torch.cuda.get_device_name(0))
print("Project:",root)


## 2. Load initial or saved weights

If training has already happened, the saved policy is loaded. Opening the notebook never resets the run. V1 replay and optimizer are not transferred: V2 has different transition rules.

In [ ]:
model_name="B"
target_steps=400000

policy,config=load_policy(model_name,root=root)
print(policy)
print("V1 pretraining steps:",config["pretraining_steps"])
print("Additional V2 target:",target_steps)
status=show_status(model_name,root=root)

### The settings and update we actually use

The replay buffer stores transitions `(state, action, reward, next_state, terminated)`. We sample a small batch to update the policy network. The target network changes less often, which makes the target values more stable.

Double DQN uses the policy network to **choose** the next action and the target network to **estimate** its value:

$$a^*=\arg\max_a Q_{policy}(s\prime,a),\qquad y=r+\gamma(1-terminated)Q_{target}(s\prime,a^*)$$

We minimize the smooth L1 loss between `Q_policy(state, action)` and this target. A true terminal state has no future value. A time-limit truncation still bootstraps. The following cells only print settings and the existing function; they do not update weights.

In [ ]:
keys=["learning_rate","gamma","batch_size","warmup_steps","train_every",
      "target_update","epsilon_start","epsilon_end","epsilon_decay_steps","memory_capacity"]
for key in keys:
    print(f"{key}: {config[key]}")

In [ ]:
import inspect
from train_v2 import train_one_step

# Show the exact implementation used by the training button.
print(inspect.getsource(train_one_step))

## 3. What V2 and the experiment change

The capture meter grows under the original close-and-slow condition. It decreases only when all police cars are at least 120 px away; a brief acceleration beside a police car holds the meter instead. Other game dynamics remain the same.

Both models use the same stable Double DQN settings and goal shaping. Half of training episodes use the original start; the other half use small changes up to 30 px and 10 degrees. This is a shared training change, described separately from the capture rule. Original and shaped episode rewards are stored separately.

The saved target is 400,000 additional steps. At 50 steps/s, training alone is about 133 minutes; evaluations and saves add time. The target is a step budget, not a hard wall-clock limit.

## 4. Start or resume explicitly

The baseline checks frozen V1 weights on V2. It may already be saved in this package. Training saves every 10,000 steps and retains snapshots every 50,000. A short diagnostic uses the original start and four fixed variations.

To stop, use **Kernel → Interrupt Kernel** and wait. Up to 10,000 steps after the last coherent checkpoint can be lost. Run this notebook again and use the same training button to resume toward the same 400k target.

Final checks require the saved 400k budget. Checkpoint selection uses the diagnostic results, with a fixed-start win first. Step 0 is eligible if later training made the policy worse. The 40 held-out test starts do not select the weights. After final tests, the same button checks the original V1 weights on the same final cases as a matched reference, without training or another video.

In [ ]:
import ipywidgets as widgets

train_button=widgets.Button(description="Start/resume V2 training",button_style="warning",layout=widgets.Layout(width="250px"))
baseline_button=widgets.Button(description="Check initial weights",layout=widgets.Layout(width="250px"))
final_button=widgets.Button(description="Final checks and video",button_style="info",layout=widgets.Layout(width="250px"))
output=widgets.Output()
buttons=[train_button,baseline_button,final_button]

def perform(function):
    # Keep earlier output visible. Persistent attempt files survive closing Jupyter.
    for button in buttons:
        button.disabled=True
    try:
        with output:
            function()
    except KeyboardInterrupt:
        with output:
            print("Stopped. The last coherent checkpoint or completed tests are saved.")
    except Exception:
        import traceback
        with output:
            traceback.print_exc()
    finally:
        for button in buttons:
            button.disabled=False

train_button.on_click(lambda _:perform(lambda:run_training(model_name,target_steps=target_steps,root=root)))
baseline_button.on_click(lambda _:perform(lambda:evaluate_initial(model_name,root=root)))
final_button.on_click(lambda _:perform(lambda:evaluate_final(model_name,root=root)))
display(widgets.VBox(buttons),output)


## 5. Read saved attempt history

Run this cell after a session or when reopening the notebook. Each attempt has its own JSON with measured training, evaluation and checkpoint time.

In [ ]:
status=show_status(model_name,root=root)
folder=root/"runs"/f"v2_{model_name.lower()}"/"attempts"
for path in sorted(folder.glob("attempt_*.json")):
    item=json.loads(path.read_text(encoding="utf-8"))
    print(path.stem,item.get("status"),"saved step",item.get("last_saved_step"),
          "training minutes",round(item.get("training_seconds",0)/60,2),
          "evaluation minutes",round(item.get("evaluation_seconds",0)/60,2))

## 6. Learning curves

These plots use the original reward so that it is comparable across runs. Training wins with random actions are not the same as frozen greedy test wins.

In [ ]:
figure=plot_training(model_name,root=root)

## 7. Saved evaluation results and video

Initial preview: 10 fixed repeats and 10 changed starts. Final checks: 10 fixed repeats and 40 new changed starts, without random actions or weight updates. After training, the original V1 weights are also tested on these same final cases in a separate reference phase. This matched reference is used for before/after comparison; the earlier preview has different changed starts. Fixed repeats represent one deterministic scene. Mean and standard deviation refer to original V2 rewards.

The report separates the total training budget from the selected checkpoint step. The selected policy can come from an earlier diagnostic snapshot.

In [ ]:
results=show_evaluations(model_name,root=root)
for result in results:
    if result.get("video"):
        display(Video(filename=str(root/result["video"]),embed=True,width=640))

## 8. Compare both models on the main laptop

After the second laptop finishes, copy its `runs/v2_b` folder into the main V2 folder without removing the original copy. Evaluate both B and C on the main laptop. Then run this cell. It requires complete final and matched-reference checks for both models, and verifies that the added training budget, game, test cases, evaluation device and package versions match.

B starts from 250k V1 steps and C from 300k. This is an adaptation comparison of two saved policies; the original equal-budget V1 experiments remain separate.

In [ ]:
try:
    comparison=compare_final(root=root)
except RuntimeError as error:
    print(error)